# Spatial drought analysis (current policies)


In [ ]:
# general
from pathlib import Path
import geopandas as gpd
import pandas as pd
import yaml 
import os

# plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

## Data

In [ ]:
# data
df = pd.read_csv("data/hydro/droughts_hydroatlas_whatif.csv", index_col=0)
df = df[df.scenario == "28C"]

# shapes/polygons
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
hydroatlas_shape["geometry"] = hydroatlas_shape["geometry"].simplify(0.01, preserve_topology=True)
hydroatlas_shape.gauge_id = hydroatlas_shape.gauge_id.astype(int)
hydroatlas_shape = hydroatlas_shape.set_index("gauge_id")
hydroatlas_shape = hydroatlas_shape.join(df)
hydroatlas_shape = hydroatlas_shape.dropna(subset=["scenario"]) # drop basins without glaciers

# main basins
main_basins = gpd.read_file(cfg["dir"]["data"] + "/HYDROATLAS/hybas_sa_lev08_v1c_dissolved.shp")
main_basins["geometry"] = main_basins["geometry"].simplify(0.01, preserve_topology=True)

## Plot (MS)

In [ ]:
# Configuration for each panel
panels_config = [
    {
        'col': 1,
        'variable': 'min_30day',
        'colorscale': 'RdYlBu',
        'zmin': -20,
        'zmax': 20,
        'colorbar_x': 0.01,
        'colorbar_title': 'Δ Min. 30-day<br>(%)'
    },
    {
        'col': 2,
        'variable': 'doy_min',
        'colorscale': 'RdYlBu',
        'zmin': -30,
        'zmax': 30,
        'colorbar_x': 0.26,
        'colorbar_title': 'Δ DOY<br>Min. flow (days)'
    },
    {
        'col': 3,
        'variable': 'intensity',
        'colorscale': 'RdYlBu_r',
        'zmin': -30,
        'zmax': 30,
        'colorbar_x': 0.51,
        'colorbar_title': 'Δ Intensity<br>(%)'
    },
    {
        'col': 4,
        'variable': 'severity',
        'colorscale': 'RdYlBu_r',
        'zmin': -70,
        'zmax': 70,
        'colorbar_x': 0.76,
        'colorbar_title': 'Δ Severity<br>(%)'
    }
]

fig = make_subplots(rows=1, cols=4, horizontal_spacing=0, 
                    specs=[[{"type": "scattergeo"}, {"type": "scattergeo"}, 
                            {"type": "scattergeo"}, {"type": "scattergeo"}]])

# Add base layer for all panels
for x in range(0, 4):
    fig.add_trace(go.Choropleth(
        geojson=eval(main_basins['geometry'].to_json()), 
        locations=main_basins.index, 
        z=main_basins.NEXT_DOWN, 
        colorscale=["rgba(255,255,255,0.0001)", "rgba(255,255,255,0.0001)"], 
        marker_line_color="rgb(210, 226, 230)", 
        showscale=False, 
        marker_line_width=0.8
    ), row=1, col=x+1)

# Add data layers using configuration
for panel in panels_config:
    colorbar = dict(
        len=0.3, 
        y=0.63, 
        ticksuffix="%" if panel['variable'] != 'doy_min' else "", 
        thickness=15, 
        outlinewidth=0.8,
        showticklabels=True,
        ticklen=5, 
        tickwidth=0.8,
        bgcolor="rgba(255,255,255, 0)",
        x=panel['colorbar_x'],
        title=panel['colorbar_title']
    )
    
    fig.add_trace(go.Choropleth(
        geojson=eval(hydroatlas_shape['geometry'].to_json()),
        locations=hydroatlas_shape.index,
        z=hydroatlas_shape[panel['variable']],
        colorscale=panel['colorscale'],
        marker_line_color='grey',
        marker_line_width=0,
        zmin=panel['zmin'],
        zmax=panel['zmax'],
        colorbar=colorbar
    ), row=1, col=panel['col'])

# Update geography settings for each subplot
geo_settings = dict(
    showframe=True, 
    framecolor="rgba(0,0,0,0.2)", 
    bgcolor="rgba(210, 226, 230, 0.7)",
    landcolor="white", 
    showland=True, 
    resolution=50,
    showlakes=True, 
    lakecolor="rgb(210, 226, 230)", 
    oceancolor="rgb(210, 226, 230)",
    showcoastlines=True, 
    coastlinecolor="rgba(0,0,0,0)",
    showcountries=True, 
    countrycolor="rgba(0,0,0,0.2)", 
    countrywidth=0.1,
    lonaxis_range=[-79, -65], 
    lataxis_range=[-56, -7]
)

# Add panel labels (a, b, c, d) to lower left of each plot
labels = ['a)', 'b)', 'c)', 'd)']
for i, label in enumerate(labels):
    fig.add_annotation(
        text=label,
        x=0.01 + i * 0.25,  # Position based on panel
        y=0.01,
        xref='paper',
        yref='paper',
        showarrow=False,
        font=dict(size=14, family='Arial'),
        xanchor='left',
        yanchor='bottom'
    )

for col in range(1, 5):
    fig.update_geos(**geo_settings, row=1, col=col)
    # Add river labels
    fig.add_trace(go.Scattergeo(
        lat=[-50.0, -47.0, -42.5, -39.0, -37.0, -33.5, -36.0, -30.0, -18.0, -9.0],
        lon=[-75.5, -75.5, -74.0, -70.0, -75.0, -73.0, -68.5, -72.5, -68.0, -76.0],
        text=["Santa<br>Cruz", "Baker", "Puelo", "Negro", "Biobío", "Maipo", "Mendoza", "Elqui", "Titicaca", "Santa"],
        mode="text",
        textfont=dict(size=10, color="#5e8588"),
        showlegend=False
    ), row=1, col=col)

fig.update_layout(width=900, height=750, margin=dict(l=5, r=5, b=5, t=5), font_family="Arial", font_size=11)

fig.write_image("figures/fig_whatif_droughts_spatial.svg", width=900, height=750)
fig.show()

## Plot (slides)

In [ ]:
# Configuration for each panel
panels_config = [
    {
        'col': 1,
        'variable': 'intensity',
        'colorscale': 'RdYlBu_r',
        'zmin': -30,
        'zmax': 30,
        'colorbar_x': 0.01,
        'colorbar_title': 'Δ Intensity<br>(%)'
    },
    {
        'col': 2,
        'variable': 'min_30day',
        'colorscale': 'RdYlBu',
        'zmin': -20,
        'zmax': 20,
        'colorbar_x': 0.21,
        'colorbar_title': 'Δ Min. 30-day<br>(%)'
    },
    {
        'col': 3,
        'variable': 'doy_min',
        'colorscale': 'RdYlBu',
        'zmin': -30,
        'zmax': 30,
        'colorbar_x': 0.41,
        'colorbar_title': 'Δ DOY<br>Min. flow (days)'
    },
    {
        'col': 4,
        'variable': 'events_per_year',
        'colorscale': 'RdYlBu_r',
        'zmin': -30,
        'zmax': 30,
        'colorbar_x': 0.61,
        'colorbar_title': 'Δ Events<br>per year (%)'
    },
    {
        'col': 5,
        'variable': 'severity',
        'colorscale': 'RdYlBu_r',
        'zmin': -70,
        'zmax': 70,
        'colorbar_x': 0.81,
        'colorbar_title': 'Δ Severity<br>(%)'
    }
]

fig = make_subplots(rows=1, cols=5, horizontal_spacing=0.01, 
                    specs=[[{"type": "scattergeo"}, {"type": "scattergeo"}, {"type": "scattergeo"},  
                            {"type": "scattergeo"}, {"type": "scattergeo"}]])

# Add base layer for all panels
for x in range(0, 5):
    fig.add_trace(go.Choropleth(
        geojson=eval(main_basins['geometry'].to_json()), 
        locations=main_basins.index, 
        z=main_basins.NEXT_DOWN, 
        colorscale=["rgba(255,255,255,0.0001)", "rgba(255,255,255,0.0001)"], 
        marker_line_color="rgb(210, 226, 230)", 
        showscale=False, 
        marker_line_width=0.8
    ), row=1, col=x+1)

# Add data layers using configuration
for panel in panels_config:
    colorbar = dict(
        len=0.3, 
        y=0.63, 
        ticksuffix="%" if panel['variable'] != 'doy_min' else "", 
        thickness=15, 
        tickwidth=1,  # Changed from 0.1 to 1
        bgcolor="rgba(255,255,255, 0)",
        x=panel['colorbar_x'],
        title=panel['colorbar_title']
    )
    
    fig.add_trace(go.Choropleth(
        geojson=eval(hydroatlas_shape['geometry'].to_json()),
        locations=hydroatlas_shape.index,
        z=hydroatlas_shape[panel['variable']],
        colorscale=panel['colorscale'],
        marker_line_color='grey',
        marker_line_width=0,
        zmin=panel['zmin'],
        zmax=panel['zmax'],
        colorbar=colorbar
    ), row=1, col=panel['col'])

# Update geography settings for each subplot
geo_settings = dict(
    showframe=True, 
    framecolor="rgba(0,0,0,0.2)", 
    bgcolor="rgba(210, 226, 230, 0.7)",
    landcolor="white", 
    showland=True, 
    resolution=50,
    showlakes=True, 
    lakecolor="rgb(210, 226, 230)", 
    oceancolor="rgb(210, 226, 230)",
    showcoastlines=True, 
    coastlinecolor="rgba(0,0,0,0)",
    showcountries=True, 
    countrycolor="rgba(0,0,0,0.2)", 
    countrywidth=0.1,
    lonaxis_range=[-79, -65], 
    lataxis_range=[-56, -7]
)

for col in range(1, 6):
    fig.update_geos(**geo_settings, row=1, col=col)
    # Add river labels
    fig.add_trace(go.Scattergeo(
        lat=[-50.0, -47.0, -42.5, -39.0, -37.0, -33.5, -36.0, -30.0, -18.0, -9.0],
        lon=[-75.5, -75.5, -74.0, -70.0, -75.0, -73.0, -68.5, -72.5, -68.0, -76.0],
        text=["Santa<br>Cruz", "Baker", "Puelo", "Negro", "Biobío", "Maipo", "Mendoza", "Elqui", "Titicaca", "Santa"],
        mode="text",
        textfont=dict(size=11, color="#5e8588"),
        showlegend=False
    ), row=1, col=col)

fig.update_layout(width=1350, height=900, margin=dict(l=5, r=5, b=5, t=5), font_family="Arial", font_size=16)

#fig.write_image("figures/fig_whatif_droughts_spatial_slide.svg", width=1350, height=900)
fig.show()